In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

In [2]:
batch = 2 # Batch
num_genes = 8 # context, aka num of genes

In [3]:
x_input = torch.from_numpy(np.random.uniform(1, 5, size=(batch, num_genes))).float() # [batch, num_genes]
total_counts = torch.from_numpy(np.random.randint(1, 10, size=(batch))).float()# [batch]

x_input.shape, x_input, total_counts.shape, total_counts

(torch.Size([2, 8]),
 tensor([[2.6142, 1.4749, 1.4445, 1.2381, 1.3521, 1.5392, 1.9505, 3.4854],
         [2.5856, 3.7566, 3.8303, 1.6913, 1.4350, 3.9516, 1.5080, 3.0375]]),
 torch.Size([2]),
 tensor([9., 6.]))

## Data Masking 
(only done for the Context/Student) We create a mask to make the target prediction task harder.  we do evaluate only the masked positions when we determine loss. We'll first generate a random distribution and then everything below our target masking threshold will be masked. For the sake fo the demonstration I'll use a lower masking ratio than our model> 

In [4]:
mask_ratio = 0.4 
rand = torch.rand(batch, num_genes)
rand

tensor([[0.5279, 0.4980, 0.2203, 0.5017, 0.4385, 0.4129, 0.5977, 0.9155],
        [0.4667, 0.8967, 0.3528, 0.4400, 0.8709, 0.6746, 0.1436, 0.5482]])

In [5]:
mask_idx = rand < mask_ratio #use probabilistic masking. it's not perfect but will work well over a large training run
mask_idx

tensor([[False, False,  True, False, False, False, False, False],
        [False, False,  True, False, False, False,  True, False]])

now we'll apply the mask.  I'll make a copy of our `x_values` so we can see how the masking changes.  

In [6]:
x_values = x_input.clone()
x_values[mask_idx] = 0.0
x_values.shape, x_values

(torch.Size([2, 8]),
 tensor([[2.6142, 1.4749, 0.0000, 1.2381, 1.3521, 1.5392, 1.9505, 3.4854],
         [2.5856, 3.7566, 0.0000, 1.6913, 1.4350, 3.9516, 0.0000, 3.0375]]))

## Forward Pass

we start by inserting in a channels dimension.  Right now we just have 1 value per gene, but we'll represent each gene with many dimensions `embed_dim` to let the model learn different combinations of gene importants.  We'll also use multiple `heads` which is basically a grouping of the embedding dimension channels so that combinations of them can learn different complex topics. 

In [7]:
embed_dim = 6
heads = 3

**Insert in the channels**

In [8]:
x = x_values.unsqueeze(-1)
x.shape, x

(torch.Size([2, 8, 1]),
 tensor([[[2.6142],
          [1.4749],
          [0.0000],
          [1.2381],
          [1.3521],
          [1.5392],
          [1.9505],
          [3.4854]],
 
         [[2.5856],
          [3.7566],
          [0.0000],
          [1.6913],
          [1.4350],
          [3.9516],
          [0.0000],
          [3.0375]]]))

### Fourier FiLM gene encoding

Instead of just relying on gene expression counts, we want to do a Fourier FiLM based projection of the gene expression counts with learnable controls on the projection.  We do this since we know that in biology, expression is typically non-linear where you have different expression plateaus including fully on or off. In this projection expression values are first sclaed by a learned gene embeddings and a scaler, then a random Fourier feature network generates expression dependent FiLM parameters (gamma, beta) to further modulate the result based on expression level. The two paths give the model both a linear signal (expression * embedding) and a nonlinear one (Fourier features -> FiLM), combined into the final gene representation. The goal is to apply the following

$\mathbf{h}_g = \alpha \, x_g \, \mathbf{e}_g \odot (1 + \boldsymbol{\gamma}_g) + \boldsymbol{\beta}_g$

where

$[\boldsymbol{\gamma}_g, \boldsymbol{\beta}_g] = \mathrm{MLP}(\phi(\alpha' x_g))$

- $x_g$ is the expression value for gene $g$
- $\mathbf{e}_g$ is the learned gene embedding
- $\alpha, \alpha'$ are learned scalars
- $\phi$ is a random Fourier feature mapping
- $\odot$ is elementwise multiplication

While we use an multi layer perceptron (MLP), the FiLM portion is specifically the $\mathbf{h}_g = x * (1 + \gamma) + \beta$ pattern. This becomes an affine transformation where gamma and beta are conditioned on some input. 

#### $\alpha$ Gene Expression Count Scaling

We'll start with our gene expression count scaler.  This will be a single value that we'll mutliply against our scaled gene expression count embeddings.   We'll first setup the learned scaler and multiply it by expression counts, after which we'll then use that to scale our gene embeddings. 

In [9]:
expr_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(expr_scaler.weight, 1.5)
expr_scaler.weight

Parameter containing:
tensor([[1.5000]], requires_grad=True)

In [10]:
scaled_x = expr_scaler(x)
scaled_x.shape, scaled_x

(torch.Size([2, 8, 1]),
 tensor([[[3.9213],
          [2.2123],
          [0.0000],
          [1.8571],
          [2.0282],
          [2.3087],
          [2.9257],
          [5.2281]],
 
         [[3.8784],
          [5.6349],
          [0.0000],
          [2.5370],
          [2.1526],
          [5.9274],
          [0.0000],
          [4.5562]]], grad_fn=<UnsafeViewBackward0>))

#### $\mathbf{e}_g$ Gene Embeddings

Now we'll initialize gene embeddings and then scale them based on our scaled expression counts. You'll see that the masking extends across all embedding channels.    

In [11]:
gene_embeddings = nn.Parameter(torch.randn(num_genes, embed_dim) * 0.02)
gene_embeddings

Parameter containing:
tensor([[ 0.0179,  0.0168, -0.0093, -0.0106, -0.0463, -0.0114],
        [ 0.0053, -0.0170,  0.0073,  0.0115, -0.0101,  0.0099],
        [ 0.0118, -0.0098,  0.0023,  0.0122,  0.0021, -0.0125],
        [ 0.0044,  0.0143, -0.0018,  0.0071,  0.0317,  0.0174],
        [-0.0100,  0.0151,  0.0240, -0.0147,  0.0220, -0.0099],
        [ 0.0258,  0.0013,  0.0159,  0.0177,  0.0358, -0.0036],
        [-0.0073,  0.0048, -0.0060,  0.0146, -0.0208, -0.0356],
        [-0.0203,  0.0154, -0.0006, -0.0014,  0.0379, -0.0340]],
       requires_grad=True)

In [12]:
scaled_x = gene_embeddings.unsqueeze(0) * scaled_x
scaled_x

tensor([[[ 0.0700,  0.0658, -0.0364, -0.0417, -0.1816, -0.0446],
         [ 0.0117, -0.0376,  0.0160,  0.0254, -0.0223,  0.0219],
         [ 0.0000, -0.0000,  0.0000,  0.0000,  0.0000, -0.0000],
         [ 0.0082,  0.0266, -0.0033,  0.0131,  0.0588,  0.0323],
         [-0.0203,  0.0307,  0.0488, -0.0299,  0.0447, -0.0201],
         [ 0.0595,  0.0030,  0.0367,  0.0410,  0.0827, -0.0083],
         [-0.0213,  0.0139, -0.0177,  0.0427, -0.0609, -0.1043],
         [-0.1061,  0.0805, -0.0031, -0.0072,  0.1983, -0.1778]],

        [[ 0.0693,  0.0651, -0.0360, -0.0412, -0.1796, -0.0441],
         [ 0.0297, -0.0958,  0.0409,  0.0648, -0.0568,  0.0557],
         [ 0.0000, -0.0000,  0.0000,  0.0000,  0.0000, -0.0000],
         [ 0.0112,  0.0363, -0.0046,  0.0179,  0.0803,  0.0441],
         [-0.0215,  0.0326,  0.0517, -0.0317,  0.0474, -0.0214],
         [ 0.1529,  0.0077,  0.0942,  0.1052,  0.2124, -0.0213],
         [-0.0000,  0.0000, -0.0000,  0.0000, -0.0000, -0.0000],
         [-0.0925,  0.0

#### $\alpha'$ Fourier Scaler 
Now we'll calculate the scaler for the fourier portion of the calculations.  To make sure the values diverge from the expression scaler, I'll use different value. 

In [13]:
fourier_input_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(fourier_input_scaler.weight, 0.5)
fourier_input_scaler.weight

Parameter containing:
tensor([[0.5000]], requires_grad=True)

In [14]:
fourier_x = fourier_input_scaler(x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 1]),
 tensor([[[1.3071],
          [0.7374],
          [0.0000],
          [0.6190],
          [0.6761],
          [0.7696],
          [0.9752],
          [1.7427]],
 
         [[1.2928],
          [1.8783],
          [0.0000],
          [0.8457],
          [0.7175],
          [1.9758],
          [0.0000],
          [1.5187]]], grad_fn=<UnsafeViewBackward0>))

#### $\phi$ Fourier Projection  
Now we'll do the fourier projection. This step projects the scaled expression counts through a fixed random matrix `fp_scaler`. This random matrix is half the size since we take the sin and cos of the result and then concatenate them together to produce a full-dimensional embedding. 

This step maps a the continuous expression level into a rich high-dimensional representation where nearby values have similar features allowing the model to learn representations from when the different plateaus of expession levels.  The `gaussian_scale` scaler is a tunable hyperparameter that can quickly improve and degrade model performance.


Since we're wroking with sine/cosine we have to think of the periodicity.  We first will take our expression and spread them across a full sin/cosine cycle, which, if you remember your trig, is $2\pi$.  This normalizes the input so that a unit change in the input corresponds to one full cycle of sin/cos. If we did not do this, the random projection matrix alone controls the frequency, increasig the fragility of the projection


*note that you'll start to see the masking disappear in this stage. Don't worry, we reapply masking at the end and, if you follow the math in this stage, each gene position only interacts with itself*

In [15]:
x_fp = (2 * np.pi * fourier_x)
x_fp

tensor([[[ 8.2128],
         [ 4.6334],
         [ 0.0000],
         [ 3.8896],
         [ 4.2478],
         [ 4.8354],
         [ 6.1276],
         [10.9497]],

        [[ 8.1228],
         [11.8018],
         [ 0.0000],
         [ 5.3135],
         [ 4.5083],
         [12.4142],
         [ 0.0000],
         [ 9.5425]]], grad_fn=<MulBackward0>)

Now we'll initiate our `fp_scaler` that will act as the scaler and then multiply it by our scaled expression data. You'll see half of the embedding dimensions get created in the broadcasted matmul. 

In [16]:
gaussian_scale = 2.0
fp_scaler = nn.Parameter(torch.randn(1, embed_dim // 2) * gaussian_scale, requires_grad=False)
fp_scaler

Parameter containing:
tensor([[0.5975, 1.5269, 0.1062]])

In [17]:
x_fp = x_fp @ fp_scaler
x_fp.shape, x_fp

(torch.Size([2, 8, 3]),
 tensor([[[ 4.9072, 12.5398,  0.8723],
          [ 2.7685,  7.0745,  0.4921],
          [ 0.0000,  0.0000,  0.0000],
          [ 2.3240,  5.9388,  0.4131],
          [ 2.5381,  6.4858,  0.4512],
          [ 2.8892,  7.3829,  0.5136],
          [ 3.6612,  9.3559,  0.6508],
          [ 6.5425, 16.7186,  1.1629]],
 
         [[ 4.8534, 12.4023,  0.8627],
          [ 7.0516, 18.0196,  1.2534],
          [ 0.0000,  0.0000,  0.0000],
          [ 3.1748,  8.1129,  0.5643],
          [ 2.6937,  6.8836,  0.4788],
          [ 7.4175, 18.9547,  1.3185],
          [ 0.0000,  0.0000,  0.0000],
          [ 5.7016, 14.5699,  1.0135]]], grad_fn=<UnsafeViewBackward0>))

**Sin/Cos**

Now we'll take the sine, cosine and concatenate them back into our `fourier_x`.

In [18]:
x_fp_sin = torch.sin(x_fp)
x_fp_sin

tensor([[[-0.9811, -0.0266,  0.7658],
         [ 0.3645,  0.7113,  0.4725],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.7295, -0.3377,  0.4015],
         [ 0.5675,  0.2012,  0.4360],
         [ 0.2498,  0.8911,  0.4913],
         [-0.4966,  0.0689,  0.6058],
         [ 0.2564, -0.8472,  0.9180]],

        [[-0.9901, -0.1633,  0.7596],
         [ 0.6950, -0.7379,  0.9501],
         [ 0.0000,  0.0000,  0.0000],
         [-0.0332,  0.9667,  0.5349],
         [ 0.4330,  0.5650,  0.4607],
         [ 0.9063,  0.1049,  0.9683],
         [ 0.0000,  0.0000,  0.0000],
         [-0.5493,  0.9078,  0.8487]]], grad_fn=<SinBackward0>)

In [19]:
x_fp_cos = torch.cos(x_fp)
x_fp_cos

tensor([[[ 0.1936,  0.9996,  0.6431],
         [-0.9312,  0.7029,  0.8813],
         [ 1.0000,  1.0000,  1.0000],
         [-0.6840,  0.9413,  0.9159],
         [-0.8233,  0.9795,  0.8999],
         [-0.9683,  0.4538,  0.8710],
         [-0.8680, -0.9976,  0.7956],
         [ 0.9666, -0.5313,  0.3966]],

        [[ 0.1405,  0.9866,  0.6504],
         [ 0.7190,  0.6749,  0.3121],
         [ 1.0000,  1.0000,  1.0000],
         [-0.9994, -0.2561,  0.8449],
         [-0.9014,  0.8251,  0.8875],
         [ 0.4227,  0.9945,  0.2496],
         [ 1.0000,  1.0000,  1.0000],
         [ 0.8356, -0.4194,  0.5289]]], grad_fn=<CosBackward0>)

In [20]:
fourier_x = torch.cat([x_fp_sin, x_fp_cos], dim=-1)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 6]),
 tensor([[[-0.9811, -0.0266,  0.7658,  0.1936,  0.9996,  0.6431],
          [ 0.3645,  0.7113,  0.4725, -0.9312,  0.7029,  0.8813],
          [ 0.0000,  0.0000,  0.0000,  1.0000,  1.0000,  1.0000],
          [ 0.7295, -0.3377,  0.4015, -0.6840,  0.9413,  0.9159],
          [ 0.5675,  0.2012,  0.4360, -0.8233,  0.9795,  0.8999],
          [ 0.2498,  0.8911,  0.4913, -0.9683,  0.4538,  0.8710],
          [-0.4966,  0.0689,  0.6058, -0.8680, -0.9976,  0.7956],
          [ 0.2564, -0.8472,  0.9180,  0.9666, -0.5313,  0.3966]],
 
         [[-0.9901, -0.1633,  0.7596,  0.1405,  0.9866,  0.6504],
          [ 0.6950, -0.7379,  0.9501,  0.7190,  0.6749,  0.3121],
          [ 0.0000,  0.0000,  0.0000,  1.0000,  1.0000,  1.0000],
          [-0.0332,  0.9667,  0.5349, -0.9994, -0.2561,  0.8449],
          [ 0.4330,  0.5650,  0.4607, -0.9014,  0.8251,  0.8875],
          [ 0.9063,  0.1049,  0.9683,  0.4227,  0.9945,  0.2496],
          [ 0.0000,  0.0000,  0.0000,  1.0000,  1

#### $\mathrm{MLP}$ Multilayer perceptron  

Now we'll add in the non-linearity learning on top of the fourier projection. This allows the model to learn additional interactions across the embeddings. This includes a linear layer that can learn how the different channels in an embeddings interact with each other, a non-linear layer (GELU), and then a final scaling up layer since we extract two components for the FiLM calculation.  Because of the GELU, you'll notice that our negative values in particular get compressed closer to 0 because of the GELU layer. 

In [21]:
film_generator = nn.Sequential(
    nn.Linear(embed_dim, embed_dim),
    nn.GELU(),
    nn.Linear(embed_dim, embed_dim * 2) # Output Gamma + Beta
)

In [22]:
fourier_x = film_generator(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 12]),
 tensor([[[-0.1567, -0.1863,  0.4832,  0.3176,  0.5937, -0.1378, -0.0266,
            0.0990,  0.4829,  0.4095,  0.2009, -0.2574],
          [ 0.1779, -0.3568,  0.1695,  0.4508,  0.1569, -0.4018, -0.1143,
           -0.1248,  0.1372,  0.2663,  0.2498, -0.3380],
          [-0.0725, -0.2520,  0.4730,  0.1840,  0.4541,  0.0805,  0.1192,
            0.0934,  0.4314,  0.3312,  0.2658, -0.1283],
          [ 0.2718, -0.3674,  0.2992,  0.4235,  0.2465, -0.2810, -0.0711,
           -0.1276,  0.1938,  0.4310,  0.1777, -0.1823],
          [ 0.2354, -0.3638,  0.2503,  0.4307,  0.2138, -0.3201, -0.0825,
           -0.1248,  0.1730,  0.3671,  0.2071, -0.2361],
          [ 0.1485, -0.3531,  0.1205,  0.4702,  0.1290, -0.4583, -0.1370,
           -0.1269,  0.1148,  0.2135,  0.2714, -0.3958],
          [ 0.0057, -0.2852,  0.0345,  0.5713,  0.3508, -0.5701, -0.0847,
           -0.1708,  0.0988,  0.1569,  0.2968, -0.3849],
          [-0.0999, -0.2009,  0.5570,  0.2718,  0.3834, -0

#### $\gamma_g, \beta_g$ FiLM components 
Now we'll split the MLP output into the two FiLM scalers.  This is just a simple matter of splitting the output of the MLP on the last dimesions.  You'll notice that we're back to our `[batch, num_genes, embed_dim]` shape now

In [23]:
gamma, beta = torch.chunk(fourier_x, 2, dim=-1)
gamma.shape, gamma, beta.shape, beta

(torch.Size([2, 8, 6]),
 tensor([[[-0.1567, -0.1863,  0.4832,  0.3176,  0.5937, -0.1378],
          [ 0.1779, -0.3568,  0.1695,  0.4508,  0.1569, -0.4018],
          [-0.0725, -0.2520,  0.4730,  0.1840,  0.4541,  0.0805],
          [ 0.2718, -0.3674,  0.2992,  0.4235,  0.2465, -0.2810],
          [ 0.2354, -0.3638,  0.2503,  0.4307,  0.2138, -0.3201],
          [ 0.1485, -0.3531,  0.1205,  0.4702,  0.1290, -0.4583],
          [ 0.0057, -0.2852,  0.0345,  0.5713,  0.3508, -0.5701],
          [-0.0999, -0.2009,  0.5570,  0.2718,  0.3834, -0.1164]],
 
         [[-0.1432, -0.1881,  0.4800,  0.3289,  0.5986, -0.1533],
          [ 0.0206, -0.2529,  0.5504,  0.2681,  0.3379, -0.0879],
          [-0.0725, -0.2520,  0.4730,  0.1840,  0.4541,  0.0805],
          [ 0.0899, -0.3406,  0.0202,  0.5306,  0.1165, -0.5822],
          [ 0.1970, -0.3591,  0.1983,  0.4420,  0.1763, -0.3707],
          [ 0.0787, -0.2927,  0.4816,  0.2624,  0.2585, -0.1127],
          [-0.0725, -0.2520,  0.4730,  0.1840,  0

#### $\mathbf{h}_g$ FiLM based gene expression respresentation 

Now we'll actually calculate the expression level embeddings for each gene based on the FiLM calcualtion.  Based on the formula you can see that the model has room to learn how much of the expression to weigh based on `scaled_x` and then how much to let the fourier projection influence through `gamma/beta`.  

In [24]:
x = scaled_x * (1.0 + gamma) + beta

x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.0325,  0.1525,  0.4290,  0.3546, -0.0885, -0.2959],
          [-0.1005, -0.1490,  0.1560,  0.3032,  0.2240, -0.3249],
          [ 0.1192,  0.0934,  0.4314,  0.3312,  0.2658, -0.1283],
          [-0.0607, -0.1108,  0.1894,  0.4497,  0.2509, -0.1591],
          [-0.1075, -0.1053,  0.2340,  0.3243,  0.2614, -0.2498],
          [-0.0686, -0.1250,  0.1559,  0.2738,  0.3648, -0.4003],
          [-0.1060, -0.1608,  0.0805,  0.2240,  0.2145, -0.4298],
          [-0.2137,  0.1222,  0.4888,  0.4399,  0.4476, -0.5341]],
 
         [[ 0.0291,  0.1381,  0.4207,  0.3631, -0.0944, -0.2925],
          [-0.0725, -0.0238,  0.5184,  0.5708,  0.0826, -0.2435],
          [ 0.1192,  0.0934,  0.4314,  0.3312,  0.2658, -0.1283],
          [-0.1618, -0.1177,  0.0651,  0.1544,  0.3976, -0.4668],
          [-0.1274, -0.1035,  0.2122,  0.2546,  0.2915, -0.3154],
          [ 0.0706,  0.0430,  0.5271,  0.5573,  0.4480, -0.3175],
          [ 0.1192,  0.0934,  0.4314,  0.3312,  0

#### Remask with learned mask

Now we need to reintroduce the masking back but, instead of a 0 value, we want to actually let the model learn a mask token.  We learn a mask token because the model needs to distinguish "this gene is masked and I need to predict it" from "this gene has zero expression." If the mask token were fixed (e.g. all zeros), it would be indistinguishable from a zero-expression gene's representation after the Fourier FiLM encoding. A learned token lets the model settle on a representation that optimally signals "predict me" to the downstream transformer blocks. Ultimately we're reapplying the masking at this point mainly so that we do our expression encoding cleanly first, and then maks after. Since our mask token is learnable, instead of a common `-1` hardcode, you'lla ctually see that they're random values.  These will change over the process of learning and can be different for each embedding dim, ultimately the model just needs a number that tells it that it's a mask. 

*As a reminder, this only happens on the context encoder, and not the target encoder during our models forward pass*

In [25]:
mask_token = nn.Parameter(torch.randn(embed_dim) * 0.02)
mask_token

Parameter containing:
tensor([-0.0275,  0.0012,  0.0180,  0.0146,  0.0195,  0.0008],
       requires_grad=True)

In [26]:
x = torch.where(mask_idx.unsqueeze(-1), mask_token, x)
x

tensor([[[ 0.0325,  0.1525,  0.4290,  0.3546, -0.0885, -0.2959],
         [-0.1005, -0.1490,  0.1560,  0.3032,  0.2240, -0.3249],
         [-0.0275,  0.0012,  0.0180,  0.0146,  0.0195,  0.0008],
         [-0.0607, -0.1108,  0.1894,  0.4497,  0.2509, -0.1591],
         [-0.1075, -0.1053,  0.2340,  0.3243,  0.2614, -0.2498],
         [-0.0686, -0.1250,  0.1559,  0.2738,  0.3648, -0.4003],
         [-0.1060, -0.1608,  0.0805,  0.2240,  0.2145, -0.4298],
         [-0.2137,  0.1222,  0.4888,  0.4399,  0.4476, -0.5341]],

        [[ 0.0291,  0.1381,  0.4207,  0.3631, -0.0944, -0.2925],
         [-0.0725, -0.0238,  0.5184,  0.5708,  0.0826, -0.2435],
         [-0.0275,  0.0012,  0.0180,  0.0146,  0.0195,  0.0008],
         [-0.1618, -0.1177,  0.0651,  0.1544,  0.3976, -0.4668],
         [-0.1274, -0.1035,  0.2122,  0.2546,  0.2915, -0.3154],
         [ 0.0706,  0.0430,  0.5271,  0.5573,  0.4480, -0.3175],
         [-0.0275,  0.0012,  0.0180,  0.0146,  0.0195,  0.0008],
         [-0.2173,  0.2

### Total count injection

Now we'll want to add the total count in. In our data prep, we normalize all cell expression counts to the same total count.  This is great since perturbSeq is relativistic in it's data, but this normalization has one drawback: cells with abnormally high or low expression totals compared to their peers lose the signal. In particular the abnormally low is our biggest concern as a perturbation that makes a cell barely viable may have very low across the board expression that gets amplified when you bring it up to our normalized count.   To avoid this we add in a learnable weight to the total expression count so that the model can learn expression levels.

The total count is multiplied across the embedding dimensions with a learnable weight and then summed to our gene expression projections. This allows the total count to act almost like a bias term. 

We start by taking our total count, injecting the embedding dimensions, and then shaping it to match our embedding counts. 

In [27]:
x_total_ct = total_counts.unsqueeze(-1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1]),
 tensor([[9.],
         [6.]]))

In [28]:
total_count_proj = nn.Linear(1, embed_dim)
nn.init.constant_(total_count_proj.weight, 0.1)
nn.init.zeros_(total_count_proj.bias)
total_count_proj.weight

Parameter containing:
tensor([[0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000]], requires_grad=True)

In [29]:
x_total_ct = total_count_proj(x_total_ct)
x_total_ct = x_total_ct.unsqueeze(1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1, 6]),
 tensor([[[0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000]],
 
         [[0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000]]],
        grad_fn=<UnsqueezeBackward0>))

### Unified representation of the cell state

Now that we have an ebedding representation of both the gene expression and total count, we're ready to sum them for a single representation of the cell state. Now it will be ready for a cell state block

In [30]:
x = x + x_total_ct
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[0.9325, 1.0525, 1.3290, 1.2546, 0.8115, 0.6041],
          [0.7995, 0.7510, 1.0560, 1.2032, 1.1240, 0.5751],
          [0.8725, 0.9012, 0.9180, 0.9146, 0.9195, 0.9008],
          [0.8393, 0.7892, 1.0894, 1.3497, 1.1509, 0.7409],
          [0.7925, 0.7947, 1.1340, 1.2243, 1.1614, 0.6502],
          [0.8314, 0.7750, 1.0559, 1.1738, 1.2648, 0.4997],
          [0.7940, 0.7392, 0.9805, 1.1240, 1.1145, 0.4702],
          [0.6863, 1.0222, 1.3888, 1.3399, 1.3476, 0.3659]],
 
         [[0.6291, 0.7381, 1.0207, 0.9631, 0.5056, 0.3075],
          [0.5275, 0.5762, 1.1184, 1.1708, 0.6826, 0.3565],
          [0.5725, 0.6012, 0.6180, 0.6146, 0.6195, 0.6008],
          [0.4382, 0.4823, 0.6651, 0.7544, 0.9976, 0.1332],
          [0.4726, 0.4965, 0.8122, 0.8546, 0.8915, 0.2846],
          [0.6706, 0.6430, 1.1271, 1.1573, 1.0480, 0.2825],
          [0.5725, 0.6012, 0.6180, 0.6146, 0.6195, 0.6008],
          [0.3827, 0.8444, 1.0738, 0.8986, 1.0983, 0.0463]]],
        gra

### Tranformer Block 
Block repeated for however many layers needed. 


```
class CellStateBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = RMSNorm(config.embed_dim)
        self.attn = BioLinearAttention(config)
        self.ln_2 = RMSNorm(config.embed_dim)
        self.mlp = SwiGLU(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x
        ```

#### RMSNorm 1

In [31]:
weight_1 = nn.Parameter(torch.ones(embed_dim))
eps_1 = 1e-6

weight_1, eps_1

(Parameter containing:
 tensor([1., 1., 1., 1., 1., 1.], requires_grad=True),
 1e-06)

In [32]:
x_fp32 = x.float()
norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + eps_1)
norm

tensor([[[0.9071, 1.0238, 1.2927, 1.2204, 0.7894, 0.5876],
         [0.8459, 0.7946, 1.1172, 1.2730, 1.1892, 0.6085],
         [0.9646, 0.9962, 1.0148, 1.0111, 1.0165, 0.9958],
         [0.8250, 0.7758, 1.0709, 1.3267, 1.1314, 0.7283],
         [0.8049, 0.8072, 1.1518, 1.2436, 1.1796, 0.6604],
         [0.8579, 0.7998, 1.0897, 1.2113, 1.3052, 0.5157],
         [0.8817, 0.8209, 1.0889, 1.2483, 1.2377, 0.5222],
         [0.6269, 0.9338, 1.2687, 1.2240, 1.2311, 0.3343]],

        [[0.8534, 1.0013, 1.3848, 1.3065, 0.6859, 0.4171],
         [0.6607, 0.7217, 1.4007, 1.4664, 0.8550, 0.4465],
         [0.9469, 0.9943, 1.0220, 1.0165, 1.0245, 0.9937],
         [0.6859, 0.7549, 1.0410, 1.1809, 1.5614, 0.2085],
         [0.6999, 0.7353, 1.2028, 1.2657, 1.3203, 0.4215],
         [0.7617, 0.7303, 1.2802, 1.3145, 1.1903, 0.3209],
         [0.9469, 0.9943, 1.0220, 1.0165, 1.0245, 0.9937],
         [0.4670, 1.0305, 1.3105, 1.0967, 1.3404, 0.0565]]],
       grad_fn=<MulBackward0>)

In [33]:
x_norm = (norm * weight_1).type_as(x)
x_norm

tensor([[[0.9071, 1.0238, 1.2927, 1.2204, 0.7894, 0.5876],
         [0.8459, 0.7946, 1.1172, 1.2730, 1.1892, 0.6085],
         [0.9646, 0.9962, 1.0148, 1.0111, 1.0165, 0.9958],
         [0.8250, 0.7758, 1.0709, 1.3267, 1.1314, 0.7283],
         [0.8049, 0.8072, 1.1518, 1.2436, 1.1796, 0.6604],
         [0.8579, 0.7998, 1.0897, 1.2113, 1.3052, 0.5157],
         [0.8817, 0.8209, 1.0889, 1.2483, 1.2377, 0.5222],
         [0.6269, 0.9338, 1.2687, 1.2240, 1.2311, 0.3343]],

        [[0.8534, 1.0013, 1.3848, 1.3065, 0.6859, 0.4171],
         [0.6607, 0.7217, 1.4007, 1.4664, 0.8550, 0.4465],
         [0.9469, 0.9943, 1.0220, 1.0165, 1.0245, 0.9937],
         [0.6859, 0.7549, 1.0410, 1.1809, 1.5614, 0.2085],
         [0.6999, 0.7353, 1.2028, 1.2657, 1.3203, 0.4215],
         [0.7617, 0.7303, 1.2802, 1.3145, 1.1903, 0.3209],
         [0.9469, 0.9943, 1.0220, 1.0165, 1.0245, 0.9937],
         [0.4670, 1.0305, 1.3105, 1.0967, 1.3404, 0.0565]]],
       grad_fn=<MulBackward0>)

#### Linear Attention

In [34]:
B, T_q, C = x_norm.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 8, 6, 2)

In [35]:
kv_input = x_norm
kv_input

tensor([[[0.9071, 1.0238, 1.2927, 1.2204, 0.7894, 0.5876],
         [0.8459, 0.7946, 1.1172, 1.2730, 1.1892, 0.6085],
         [0.9646, 0.9962, 1.0148, 1.0111, 1.0165, 0.9958],
         [0.8250, 0.7758, 1.0709, 1.3267, 1.1314, 0.7283],
         [0.8049, 0.8072, 1.1518, 1.2436, 1.1796, 0.6604],
         [0.8579, 0.7998, 1.0897, 1.2113, 1.3052, 0.5157],
         [0.8817, 0.8209, 1.0889, 1.2483, 1.2377, 0.5222],
         [0.6269, 0.9338, 1.2687, 1.2240, 1.2311, 0.3343]],

        [[0.8534, 1.0013, 1.3848, 1.3065, 0.6859, 0.4171],
         [0.6607, 0.7217, 1.4007, 1.4664, 0.8550, 0.4465],
         [0.9469, 0.9943, 1.0220, 1.0165, 1.0245, 0.9937],
         [0.6859, 0.7549, 1.0410, 1.1809, 1.5614, 0.2085],
         [0.6999, 0.7353, 1.2028, 1.2657, 1.3203, 0.4215],
         [0.7617, 0.7303, 1.2802, 1.3145, 1.1903, 0.3209],
         [0.9469, 0.9943, 1.0220, 1.0165, 1.0245, 0.9937],
         [0.4670, 1.0305, 1.3105, 1.0967, 1.3404, 0.0565]]],
       grad_fn=<MulBackward0>)

In [36]:
T_kv = kv_input.size(1)
T_kv

8

In [37]:
q_proj = nn.Linear(embed_dim, embed_dim)
q_proj.weight

Parameter containing:
tensor([[-0.0231,  0.2431,  0.2565, -0.0081,  0.1238, -0.0852],
        [ 0.3037,  0.1781, -0.1862, -0.4054,  0.3526,  0.3653],
        [ 0.3399,  0.0708, -0.3287, -0.1024,  0.0835,  0.0267],
        [-0.3296,  0.3027, -0.0604,  0.0283,  0.3750,  0.1366],
        [ 0.1412,  0.2370, -0.2736,  0.2711,  0.2046, -0.3831],
        [-0.0375,  0.0486,  0.1879, -0.2293, -0.0825,  0.3745]],
       requires_grad=True)

In [38]:
q = q_proj(x_norm).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 3, 8, 2]),
 tensor([[[[ 0.8881,  0.0610],
           [ 0.8361,  0.1615],
           [ 0.8038,  0.4393],
           [ 0.8023,  0.1621],
           [ 0.8435,  0.1724],
           [ 0.8528,  0.2033],
           [ 0.8479,  0.1780],
           [ 0.9427,  0.0261]],
 
          [[-0.3367,  0.7235],
           [-0.2874,  0.8392],
           [-0.1764,  0.8480],
           [-0.2878,  0.8394],
           [-0.3082,  0.8571],
           [-0.2604,  0.8676],
           [-0.2598,  0.8428],
           [-0.4006,  0.9212]],
 
          [[ 0.5590,  0.5359],
           [ 0.6321,  0.4568],
           [ 0.4699,  0.6623],
           [ 0.5942,  0.4853],
           [ 0.5900,  0.4924],
           [ 0.6851,  0.4213],
           [ 0.6874,  0.4208],
           [ 0.6930,  0.4054]]],
 
 
         [[[ 0.9085, -0.1101],
           [ 0.8662, -0.2159],
           [ 0.8067,  0.4321],
           [ 0.8915,  0.1426],
           [ 0.8792,  0.0716],
           [ 0.8885, -0.0273],
           [ 0.8067,  0.4321],


In [39]:
k_proj = nn.Linear(embed_dim, embed_dim)
k_proj.weight

Parameter containing:
tensor([[-0.2497, -0.3449, -0.0961, -0.0312,  0.2019, -0.0870],
        [-0.2190, -0.2864, -0.1430, -0.1487, -0.1879, -0.0004],
        [ 0.1072, -0.3167,  0.2405, -0.1787,  0.2222, -0.3549],
        [ 0.2844,  0.3255,  0.2145,  0.3549, -0.1167, -0.1732],
        [ 0.0305, -0.3355, -0.0176, -0.3621, -0.1592, -0.0873],
        [-0.1152,  0.0465,  0.2909, -0.0470, -0.3589, -0.2138]],
       requires_grad=True)

In [40]:
k = k_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 3, 8, 2]),
 tensor([[[[-0.6951, -0.7486],
           [-0.5066, -0.7274],
           [-0.6563, -0.7253],
           [-0.5142, -0.7080],
           [-0.5095, -0.7209],
           [-0.4753, -0.7402],
           [-0.5038, -0.7442],
           [-0.4806, -0.7415]],
 
          [[ 0.0213,  1.3495],
           [ 0.1171,  1.1883],
           [-0.0877,  1.1258],
           [ 0.0447,  1.1713],
           [ 0.1017,  1.1698],
           [ 0.1799,  1.1681],
           [ 0.1516,  1.2015],
           [ 0.2014,  1.2290]],
 
          [[-0.7167, -0.4473],
           [-0.7231, -0.6524],
           [-0.6968, -0.6950],
           [-0.7373, -0.6717],
           [-0.7215, -0.6433],
           [-0.7120, -0.6805],
           [-0.7216, -0.6613],
           [-0.7441, -0.5307]]],
 
 
         [[[-0.6915, -0.7369],
           [-0.5219, -0.6724],
           [-0.6502, -0.7242],
           [-0.3328, -0.7262],
           [-0.4149, -0.7142],
           [-0.4551, -0.7201],
           [-0.6502, -0.7242],


In [41]:
v_proj = nn.Linear(embed_dim, embed_dim)
v_proj.weight

Parameter containing:
tensor([[ 0.1890,  0.1932, -0.3770, -0.4035, -0.2015, -0.0234],
        [ 0.1834,  0.3926,  0.3323, -0.2823,  0.1848, -0.0502],
        [-0.0995,  0.3710,  0.1958, -0.3768,  0.0212, -0.1141],
        [ 0.2239,  0.1344, -0.1737, -0.3258, -0.1150,  0.3369],
        [ 0.2333,  0.0494, -0.0830,  0.1959, -0.2787, -0.2750],
        [ 0.3503,  0.3609,  0.0180,  0.0586, -0.1674, -0.0650]],
       requires_grad=True)

In [42]:
v = v_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 3, 8, 2]),
 tensor([[[[-1.1685,  0.9526],
           [-1.2605,  0.8511],
           [-1.0291,  0.9405],
           [-1.2634,  0.7926],
           [-1.2662,  0.8639],
           [-1.2431,  0.8897],
           [-1.2357,  0.8788],
           [-1.3143,  0.9512]],
 
          [[ 0.3552, -0.1046],
           [ 0.2282, -0.1747],
           [ 0.3219,  0.1325],
           [ 0.1790, -0.1443],
           [ 0.2486, -0.1600],
           [ 0.2598, -0.1910],
           [ 0.2490, -0.1848],
           [ 0.3819, -0.3125]],
 
          [[ 0.0545,  0.8088],
           [-0.0634,  0.6363],
           [-0.1269,  0.7372],
           [-0.0716,  0.6264],
           [-0.0925,  0.6236],
           [-0.0769,  0.6249],
           [-0.0460,  0.6539],
           [-0.0660,  0.6205]]],
 
 
         [[[-1.2276,  0.9296],
           [-1.4233,  0.7744],
           [-1.0393,  0.9390],
           [-1.2981,  0.8957],
           [-1.3509,  0.8651],
           [-1.3605,  0.8675],
           [-1.0393,  0.9390],


In [43]:
q = F.elu(q) + 1.0
q

tensor([[[[1.8881, 1.0610],
          [1.8361, 1.1615],
          [1.8038, 1.4393],
          [1.8023, 1.1621],
          [1.8435, 1.1724],
          [1.8528, 1.2033],
          [1.8479, 1.1780],
          [1.9427, 1.0261]],

         [[0.7141, 1.7235],
          [0.7502, 1.8392],
          [0.8383, 1.8480],
          [0.7499, 1.8394],
          [0.7348, 1.8571],
          [0.7707, 1.8676],
          [0.7712, 1.8428],
          [0.6699, 1.9212]],

         [[1.5590, 1.5359],
          [1.6321, 1.4568],
          [1.4699, 1.6623],
          [1.5942, 1.4853],
          [1.5900, 1.4924],
          [1.6851, 1.4213],
          [1.6874, 1.4208],
          [1.6930, 1.4054]]],


        [[[1.9085, 0.8957],
          [1.8662, 0.8058],
          [1.8067, 1.4321],
          [1.8915, 1.1426],
          [1.8792, 1.0716],
          [1.8885, 0.9731],
          [1.8067, 1.4321],
          [2.0189, 0.9760]],

         [[0.6645, 1.6692],
          [0.6061, 1.7191],
          [0.8312, 1.8557],
          

In [44]:
k = F.elu(k) + 1.0
k

tensor([[[[0.4990, 0.4730],
          [0.6026, 0.4831],
          [0.5188, 0.4842],
          [0.5980, 0.4926],
          [0.6008, 0.4863],
          [0.6217, 0.4770],
          [0.6042, 0.4751],
          [0.6184, 0.4764]],

         [[1.0213, 2.3495],
          [1.1171, 2.1883],
          [0.9160, 2.1258],
          [1.0447, 2.1713],
          [1.1017, 2.1698],
          [1.1799, 2.1681],
          [1.1516, 2.2015],
          [1.2014, 2.2290]],

         [[0.4883, 0.6394],
          [0.4852, 0.5208],
          [0.4982, 0.4991],
          [0.4784, 0.5108],
          [0.4860, 0.5256],
          [0.4906, 0.5064],
          [0.4860, 0.5162],
          [0.4751, 0.5882]]],


        [[[0.5008, 0.4786],
          [0.5934, 0.5105],
          [0.5219, 0.4847],
          [0.7169, 0.4837],
          [0.6604, 0.4896],
          [0.6344, 0.4867],
          [0.5219, 0.4847],
          [0.6519, 0.4763]],

         [[1.0669, 2.4189],
          [1.1372, 2.3084],
          [0.9179, 2.1230],
          

In [45]:
kv_matmul = k.transpose(-2, -1) @ v
kv_matmul.shape, kv_matmul

(torch.Size([2, 3, 2, 2]),
 tensor([[[[-5.7250,  4.1415],
           [-4.7044,  3.4229]],
 
          [[ 2.4257, -1.3213],
           [ 4.9093, -2.5244]],
 
          [[-0.2380,  2.5929],
           [-0.2482,  2.8830]]],
 
 
         [[[-6.0813,  4.3495],
           [-4.8950,  3.5267]],
 
          [[ 2.9619, -2.2092],
           [ 5.6780, -3.8644]],
 
          [[-0.1942,  2.5529],
           [-0.1775,  3.0080]]]], grad_fn=<UnsafeViewBackward0>))

In [46]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 3, 2, 1]),
 tensor([[[[ 4.6634],
           [ 3.8478]],
 
          [[ 8.7337],
           [17.6033]],
 
          [[ 3.8879],
           [ 4.3064]]],
 
 
         [[[ 4.8016],
           [ 3.8948]],
 
          [[ 9.1847],
           [17.6972]],
 
          [[ 3.9404],
           [ 4.6153]]]], grad_fn=<UnsqueezeBackward0>))

In [47]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 3, 8, 1]),
 tensor([[[[0.0776],
           [0.0767],
           [0.0717],
           [0.0777],
           [0.0763],
           [0.0754],
           [0.0760],
           [0.0769]],
 
          [[0.0273],
           [0.0257],
           [0.0251],
           [0.0257],
           [0.0256],
           [0.0252],
           [0.0255],
           [0.0252]],
 
          [[0.0789],
           [0.0792],
           [0.0777],
           [0.0794],
           [0.0793],
           [0.0789],
           [0.0789],
           [0.0791]]],
 
 
         [[[0.0790],
           [0.0827],
           [0.0702],
           [0.0739],
           [0.0758],
           [0.0778],
           [0.0702],
           [0.0741]],
 
          [[0.0281],
           [0.0278],
           [0.0247],
           [0.0240],
           [0.0251],
           [0.0262],
           [0.0247],
           [0.0242]],
 
          [[0.0764],
           [0.0783],
           [0.0743],
           [0.0761],
           [0.0768],
          

In [48]:
y = (q @ kv_matmul) * z
y.shape, y

(torch.Size([2, 3, 8, 2]),
 tensor([[[[-1.2260,  0.8885],
           [-1.2259,  0.8886],
           [-1.2256,  0.8887],
           [-1.2259,  0.8886],
           [-1.2259,  0.8886],
           [-1.2259,  0.8886],
           [-1.2259,  0.8886],
           [-1.2261,  0.8885]],
 
          [[ 0.2787, -0.1447],
           [ 0.2787, -0.1447],
           [ 0.2787, -0.1449],
           [ 0.2787, -0.1447],
           [ 0.2787, -0.1447],
           [ 0.2787, -0.1447],
           [ 0.2787, -0.1448],
           [ 0.2787, -0.1446]],
 
          [[-0.0593,  0.6682],
           [-0.0594,  0.6682],
           [-0.0592,  0.6683],
           [-0.0594,  0.6682],
           [-0.0594,  0.6682],
           [-0.0595,  0.6681],
           [-0.0595,  0.6681],
           [-0.0595,  0.6681]]],
 
 
         [[[-1.2638,  0.9057],
           [-1.2640,  0.9057],
           [-1.2627,  0.9057],
           [-1.2633,  0.9057],
           [-1.2634,  0.9057],
           [-1.2636,  0.9057],
           [-1.2627,  0.9057],


In [49]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 8, 6]),
 tensor([[[-1.2260,  0.8885,  0.2787, -0.1447, -0.0593,  0.6682],
          [-1.2259,  0.8886,  0.2787, -0.1447, -0.0594,  0.6682],
          [-1.2256,  0.8887,  0.2787, -0.1449, -0.0592,  0.6683],
          [-1.2259,  0.8886,  0.2787, -0.1447, -0.0594,  0.6682],
          [-1.2259,  0.8886,  0.2787, -0.1447, -0.0594,  0.6682],
          [-1.2259,  0.8886,  0.2787, -0.1447, -0.0595,  0.6681],
          [-1.2259,  0.8886,  0.2787, -0.1448, -0.0595,  0.6681],
          [-1.2261,  0.8885,  0.2787, -0.1446, -0.0595,  0.6681]],
 
         [[-1.2638,  0.9057,  0.3211, -0.2222, -0.0436,  0.6499],
          [-1.2640,  0.9057,  0.3211, -0.2218, -0.0437,  0.6499],
          [-1.2627,  0.9057,  0.3212, -0.2225, -0.0431,  0.6501],
          [-1.2633,  0.9057,  0.3211, -0.2220, -0.0444,  0.6496],
          [-1.2634,  0.9057,  0.3211, -0.2219, -0.0439,  0.6498],
          [-1.2636,  0.9057,  0.3211, -0.2220, -0.0440,  0.6498],
          [-1.2627,  0.9057,  0.3212, -0.2225, -0

In [50]:
gate = nn.Linear(embed_dim, embed_dim)
gate.weight

Parameter containing:
tensor([[-1.8022e-01, -1.1732e-01,  3.4852e-01, -3.9222e-01, -3.1258e-01,
          1.9080e-01],
        [ 2.3373e-01, -4.0595e-01,  2.2695e-01,  7.0670e-02, -1.0169e-01,
          3.1187e-01],
        [-1.9505e-01,  7.6476e-02, -2.9354e-01,  3.6273e-01,  2.1643e-01,
          2.8700e-01],
        [ 3.4000e-01,  7.3450e-02,  3.4441e-01, -3.1288e-01,  3.6564e-01,
          2.2021e-01],
        [ 3.4670e-04, -3.0798e-01, -1.9036e-02, -1.6700e-01,  2.8734e-01,
         -3.3133e-01],
        [ 8.7339e-02, -1.6832e-01, -1.6680e-01,  3.8195e-01,  3.2399e-01,
          3.5661e-01]], requires_grad=True)

In [51]:
y = torch.sigmoid(gate(x_norm)) * y
y.shape, y

(torch.Size([2, 8, 6]),
 tensor([[[-0.5868,  0.5617,  0.1377, -0.1119, -0.0279,  0.4043],
          [-0.5366,  0.5635,  0.1487, -0.1128, -0.0305,  0.4384],
          [-0.5821,  0.5722,  0.1488, -0.1159, -0.0275,  0.4348],
          [-0.5395,  0.5715,  0.1527, -0.1119, -0.0296,  0.4464],
          [-0.5494,  0.5652,  0.1488, -0.1132, -0.0302,  0.4373],
          [-0.5239,  0.5529,  0.1475, -0.1137, -0.0316,  0.4362],
          [-0.5242,  0.5547,  0.1473, -0.1131, -0.0311,  0.4352],
          [-0.5457,  0.5284,  0.1433, -0.1116, -0.0315,  0.4120]],
 
         [[-0.6082,  0.5685,  0.1540, -0.1681, -0.0207,  0.3810],
          [-0.5967,  0.5843,  0.1632, -0.1652, -0.0218,  0.4049],
          [-0.5999,  0.5826,  0.1718, -0.1779, -0.0201,  0.4231],
          [-0.5070,  0.5296,  0.1700, -0.1728, -0.0257,  0.4185],
          [-0.5497,  0.5610,  0.1690, -0.1723, -0.0238,  0.4198],
          [-0.5556,  0.5651,  0.1630, -0.1709, -0.0238,  0.4099],
          [-0.5999,  0.5826,  0.1718, -0.1779, -0

In [52]:
c_proj = nn.Linear(embed_dim, embed_dim)
c_proj.weight

Parameter containing:
tensor([[ 0.3939,  0.0182,  0.1997,  0.0595, -0.2500,  0.1900],
        [-0.3856,  0.2702, -0.2797,  0.2497, -0.2538, -0.3344],
        [-0.0448,  0.3940, -0.1179,  0.0569,  0.0272, -0.0731],
        [-0.1225, -0.0646, -0.2747,  0.2002,  0.0400, -0.3685],
        [-0.1828,  0.3875,  0.3130,  0.2220, -0.3225, -0.1921],
        [ 0.3245, -0.3055,  0.0726, -0.0164,  0.2794,  0.3420]],
       requires_grad=True)

In [53]:
x_attn = c_proj(y)
x_attn.shape, x_attn

(torch.Size([2, 8, 6]),
 tensor([[[-0.0684,  0.5736, -0.0359, -0.5247,  0.5147, -0.5106],
          [-0.0394,  0.5407, -0.0413, -0.5468,  0.5037, -0.4831],
          [-0.0587,  0.5602, -0.0357, -0.5410,  0.5145, -0.5009],
          [-0.0382,  0.5402, -0.0390, -0.5508,  0.5070, -0.4832],
          [-0.0447,  0.5462, -0.0400, -0.5450,  0.5068, -0.4881],
          [-0.0350,  0.5341, -0.0458, -0.5468,  0.4975, -0.4769],
          [-0.0354,  0.5350, -0.0450, -0.5463,  0.4984, -0.4778],
          [-0.0493,  0.5456, -0.0521, -0.5320,  0.4957, -0.4851]],
 
         [[-0.0830,  0.5710, -0.0355, -0.5294,  0.5160, -0.5235],
          [-0.0714,  0.5613, -0.0324, -0.5426,  0.5193, -0.5161],
          [-0.0687,  0.5500, -0.0360, -0.5536,  0.5150, -0.5091],
          [-0.0326,  0.5046, -0.0603, -0.5586,  0.4808, -0.4661],
          [-0.0492,  0.5290, -0.0460, -0.5554,  0.4997, -0.4887],
          [-0.0545,  0.5377, -0.0426, -0.5494,  0.5027, -0.4956],
          [-0.0687,  0.5500, -0.0360, -0.5536,  0

#### Residual Connection

In [54]:
x = x + x_attn
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.8641,  1.6261,  1.2931,  0.7299,  1.3262,  0.0935],
          [ 0.7601,  1.2917,  1.0146,  0.6564,  1.6277,  0.0920],
          [ 0.8138,  1.4614,  0.8823,  0.3736,  1.4340,  0.3999],
          [ 0.8011,  1.3294,  1.0504,  0.7989,  1.6579,  0.2577],
          [ 0.7478,  1.3409,  1.0939,  0.6793,  1.6681,  0.1621],
          [ 0.7964,  1.3091,  1.0101,  0.6270,  1.7623,  0.0228],
          [ 0.7586,  1.2742,  0.9355,  0.5777,  1.6129, -0.0075],
          [ 0.6369,  1.5678,  1.3366,  0.8079,  1.8433, -0.1191]],
 
         [[ 0.5460,  1.3091,  0.9853,  0.4337,  1.0216, -0.2160],
          [ 0.4561,  1.1375,  1.0859,  0.6282,  1.2019, -0.1596],
          [ 0.5038,  1.1511,  0.5820,  0.0610,  1.1345,  0.0918],
          [ 0.4057,  0.9869,  0.6047,  0.1958,  1.4784, -0.3329],
          [ 0.4234,  1.0255,  0.7662,  0.2992,  1.3912, -0.2041],
          [ 0.6161,  1.1807,  1.0845,  0.6079,  1.5507, -0.2131],
          [ 0.5038,  1.1511,  0.5820,  0.0610,  1

#### RMSNorm 2

In [55]:
weight_2 = nn.Parameter(torch.ones(embed_dim))
eps_2 = 1e-6

weight_2, eps_2

(Parameter containing:
 tensor([1., 1., 1., 1., 1., 1.], requires_grad=True),
 1e-06)

In [56]:
x_fp32 = x.float()
norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + eps_2)
norm

tensor([[[ 0.7800,  1.4679,  1.1673,  0.6589,  1.1972,  0.0844],
         [ 0.7380,  1.2541,  0.9852,  0.6373,  1.5804,  0.0893],
         [ 0.8185,  1.4697,  0.8873,  0.3758,  1.4421,  0.4022],
         [ 0.7435,  1.2338,  0.9749,  0.7415,  1.5387,  0.2392],
         [ 0.7012,  1.2573,  1.0257,  0.6370,  1.5641,  0.1520],
         [ 0.7444,  1.2236,  0.9442,  0.5861,  1.6472,  0.0213],
         [ 0.7580,  1.2732,  0.9348,  0.5773,  1.6116, -0.0075],
         [ 0.5285,  1.3008,  1.1091,  0.6703,  1.5295, -0.0989]],

        [[ 0.6479,  1.5534,  1.1692,  0.5146,  1.2122, -0.2563],
         [ 0.5240,  1.3068,  1.2476,  0.7217,  1.3808, -0.1834],
         [ 0.6880,  1.5721,  0.7948,  0.0833,  1.5494,  0.1253],
         [ 0.5071,  1.2338,  0.7560,  0.2448,  1.8482, -0.4161],
         [ 0.5262,  1.2745,  0.9523,  0.3718,  1.7290, -0.2536],
         [ 0.6283,  1.2040,  1.1060,  0.6199,  1.5814, -0.2173],
         [ 0.6880,  1.5721,  0.7948,  0.0833,  1.5494,  0.1253],
         [ 0.3221,  1.3

In [57]:
x_norm2 = (norm * weight_2).type_as(x)
x_norm2

tensor([[[ 0.7800,  1.4679,  1.1673,  0.6589,  1.1972,  0.0844],
         [ 0.7380,  1.2541,  0.9852,  0.6373,  1.5804,  0.0893],
         [ 0.8185,  1.4697,  0.8873,  0.3758,  1.4421,  0.4022],
         [ 0.7435,  1.2338,  0.9749,  0.7415,  1.5387,  0.2392],
         [ 0.7012,  1.2573,  1.0257,  0.6370,  1.5641,  0.1520],
         [ 0.7444,  1.2236,  0.9442,  0.5861,  1.6472,  0.0213],
         [ 0.7580,  1.2732,  0.9348,  0.5773,  1.6116, -0.0075],
         [ 0.5285,  1.3008,  1.1091,  0.6703,  1.5295, -0.0989]],

        [[ 0.6479,  1.5534,  1.1692,  0.5146,  1.2122, -0.2563],
         [ 0.5240,  1.3068,  1.2476,  0.7217,  1.3808, -0.1834],
         [ 0.6880,  1.5721,  0.7948,  0.0833,  1.5494,  0.1253],
         [ 0.5071,  1.2338,  0.7560,  0.2448,  1.8482, -0.4161],
         [ 0.5262,  1.2745,  0.9523,  0.3718,  1.7290, -0.2536],
         [ 0.6283,  1.2040,  1.1060,  0.6199,  1.5814, -0.2173],
         [ 0.6880,  1.5721,  0.7948,  0.0833,  1.5494,  0.1253],
         [ 0.3221,  1.3

#### SwiGLU MLP

In [58]:
def _make_divisible(v, divisor=4):
    return max(divisor, int(v + divisor / 2) // divisor * divisor)

In [59]:
mlp_ratio = 2.0
hidden_dim = _make_divisible(int(embed_dim * mlp_ratio * 2 / 3))
hidden_dim

8

In [60]:
w1 = nn.Linear(embed_dim, hidden_dim, bias=False)
w1.weight

Parameter containing:
tensor([[ 0.0153,  0.0333,  0.3471, -0.1081, -0.2297, -0.0047],
        [-0.2948,  0.2273, -0.0235, -0.1824,  0.3430, -0.2176],
        [ 0.1398, -0.2329,  0.2118,  0.3192,  0.1116,  0.3732],
        [-0.0092, -0.4013, -0.3164,  0.2312,  0.3366, -0.2266],
        [ 0.1875, -0.1448,  0.0153,  0.4074,  0.0394, -0.1542],
        [-0.1572, -0.2613,  0.3953, -0.0701,  0.0298,  0.0370],
        [-0.2622, -0.4079,  0.2502,  0.0391, -0.0414,  0.3298],
        [ 0.0875,  0.0131, -0.1836, -0.1688, -0.0617,  0.1468]],
       requires_grad=True)

In [61]:
xw1 = w1(x_norm2)
xw1.shape, xw1

(torch.Size([2, 8, 8]),
 tensor([[[ 1.1945e-01,  3.4832e-01,  3.8977e-01, -4.2942e-01,  2.5418e-01,
           -5.2138e-02, -5.0731e-01, -2.9964e-01],
          [-3.7247e-02,  4.5071e-01,  4.3283e-01, -1.6277e-01,  2.8003e-01,
           -4.8571e-02, -4.6978e-01, -2.9194e-01],
          [-4.2225e-03,  4.1048e-01,  3.9100e-01, -3.9696e-01,  1.0213e-01,
           -1.3042e-01, -5.0459e-01, -1.6551e-01],
          [-4.3776e-02,  3.7880e-01,  5.2068e-01, -1.7531e-01,  3.0153e-01,
           -5.1154e-02, -4.1026e-01, -2.8286e-01],
          [-2.0140e-02,  4.4218e-01,  4.5698e-01, -1.9631e-01,  2.6283e-01,
           -2.5713e-02, -4.2994e-01, -2.9231e-01],
          [-6.1889e-02,  4.8992e-01,  3.9787e-01, -1.1155e-01,  2.7726e-01,
           -5.4729e-02, -4.9643e-01, -2.8971e-01],
          [-5.4000e-02,  4.9305e-01,  3.6870e-01, -1.3610e-01,  2.7196e-01,
           -7.5039e-02, -5.3096e-01, -2.8671e-01],
          [ 1.3111e-02,  5.3762e-01,  3.5353e-01, -1.8565e-01,  2.7635e-01,
           

In [62]:
w2 = nn.Linear(embed_dim, hidden_dim, bias=False)
w2.weight

Parameter containing:
tensor([[ 0.0247,  0.2295,  0.3978,  0.1295, -0.2310, -0.0311],
        [-0.3207, -0.3454,  0.3570, -0.1645, -0.3339,  0.3382],
        [ 0.0801, -0.2790, -0.3152,  0.3262,  0.3525,  0.3178],
        [-0.2004,  0.2700,  0.1366, -0.0191,  0.0424, -0.3020],
        [ 0.2464, -0.2683, -0.0468,  0.3531, -0.0050, -0.3263],
        [-0.1121,  0.1955, -0.0291, -0.3075, -0.0460, -0.0941],
        [-0.1379, -0.0648,  0.1161,  0.0060, -0.0965, -0.2094],
        [ 0.0975,  0.2843, -0.1034,  0.1658,  0.0891,  0.2719]],
       requires_grad=True)

In [63]:
xw2 = w2(x_norm2)
xw2.shape, xw2

(torch.Size([2, 8, 8]),
 tensor([[[ 6.2666e-01, -8.2005e-01, -5.1112e-02,  4.1217e-01, -5.7003e-02,
           -1.0002e-01, -1.9636e-01,  6.1150e-01],
          [ 4.1265e-01, -9.2056e-01,  1.9217e-01,  3.5314e-01, -1.2616e-02,
           -1.4325e-01, -2.3602e-01,  5.9741e-01],
          [ 4.1353e-01, -8.6072e-01,  1.3470e-01,  2.8652e-01, -2.3984e-01,
           -4.9947e-02, -3.2621e-01,  7.0606e-01],
          [ 4.2249e-01, -8.7147e-01,  2.6842e-01,  2.9614e-01, -1.7232e-02,
           -1.9175e-01, -2.6338e-01,  6.4750e-01],
          [ 4.3038e-01, -8.6867e-01,  1.8960e-01,  3.4732e-01, -4.4945e-02,
           -1.4472e-01, -2.3800e-01,  6.0606e-01],
          [ 3.6953e-01, -9.6355e-01,  1.9935e-01,  3.6237e-01,  2.8154e-03,
           -1.2966e-01, -2.3219e-01,  5.7256e-01],
          [ 3.8552e-01, -9.8480e-01,  1.6495e-01,  3.7912e-01, -1.9410e-04,
           -1.1418e-01, -2.2894e-01,  5.7647e-01],
          [ 4.8937e-01, -8.7730e-01,  5.6313e-02,  4.7870e-01, -9.2475e-03,
           

In [64]:
xw1 = F.silu(xw1)
xw1.shape, xw1

(torch.Size([2, 8, 8]),
 tensor([[[ 6.3288e-02,  2.0419e-01,  2.3239e-01, -1.6930e-01,  1.4316e-01,
           -2.5390e-02, -1.9066e-01, -1.2754e-01],
          [-1.8277e-02,  2.7530e-01,  2.6253e-01, -7.4774e-02,  1.5949e-01,
           -2.3696e-02, -1.8071e-01, -1.2481e-01],
          [-2.1068e-03,  2.4678e-01,  2.3324e-01, -1.5960e-01,  5.3671e-02,
           -6.0964e-02, -1.8996e-01, -7.5924e-02],
          [-2.1409e-02,  2.2485e-01,  3.2662e-01, -7.9991e-02,  1.7332e-01,
           -2.4923e-02, -1.6363e-01, -1.2156e-01],
          [-9.9687e-03,  2.6919e-01,  2.7981e-01, -8.8550e-02,  1.4859e-01,
           -1.2691e-02, -1.6946e-01, -1.2494e-01],
          [-2.9987e-02,  3.0379e-01,  2.3799e-01, -5.2667e-02,  1.5772e-01,
           -2.6616e-02, -1.8784e-01, -1.2402e-01],
          [-2.6271e-02,  3.0610e-01,  2.1795e-01, -6.3425e-02,  1.5436e-01,
           -3.6112e-02, -1.9661e-01, -1.2294e-01],
          [ 6.5985e-03,  3.3938e-01,  2.0769e-01, -8.4232e-02,  1.5715e-01,
           

In [65]:
xw = xw1 * xw2
xw.shape, xw

(torch.Size([2, 8, 8]),
 tensor([[[ 3.9660e-02, -1.6745e-01, -1.1878e-02, -6.9783e-02, -8.1604e-03,
            2.5395e-03,  3.7439e-02, -7.7992e-02],
          [-7.5418e-03, -2.5343e-01,  5.0451e-02, -2.6406e-02, -2.0122e-03,
            3.3945e-03,  4.2652e-02, -7.4565e-02],
          [-8.7123e-04, -2.1241e-01,  3.1418e-02, -4.5727e-02, -1.2872e-02,
            3.0450e-03,  6.1966e-02, -5.3607e-02],
          [-9.0452e-03, -1.9595e-01,  8.7671e-02, -2.3689e-02, -2.9868e-03,
            4.7790e-03,  4.3097e-02, -7.8711e-02],
          [-4.2903e-03, -2.3384e-01,  5.3053e-02, -3.0755e-02, -6.6782e-03,
            1.8367e-03,  4.0330e-02, -7.5724e-02],
          [-1.1081e-02, -2.9272e-01,  4.7443e-02, -1.9085e-02,  4.4405e-04,
            3.4512e-03,  4.3615e-02, -7.1007e-02],
          [-1.0128e-02, -3.0144e-01,  3.5952e-02, -2.4046e-02, -2.9961e-05,
            4.1231e-03,  4.5011e-02, -7.0874e-02],
          [ 3.2291e-03, -2.9774e-01,  1.1696e-02, -4.0321e-02, -1.4532e-03,
           

In [66]:
w3 = nn.Linear(hidden_dim, embed_dim, bias=False)
w3.weight

Parameter containing:
tensor([[-0.1814,  0.0344, -0.1780,  0.0234,  0.3139, -0.2499, -0.2287,  0.0247],
        [ 0.1713, -0.0545, -0.0391,  0.2580, -0.1000, -0.2788, -0.0403,  0.2863],
        [ 0.1853, -0.1502,  0.1660,  0.1794,  0.0857, -0.3516,  0.2187,  0.2536],
        [-0.2732,  0.0620,  0.1952,  0.2694,  0.3514, -0.0092,  0.1931,  0.0488],
        [-0.0400, -0.2687,  0.3494,  0.2763,  0.3439, -0.1924,  0.2887, -0.3236],
        [ 0.1998, -0.0029, -0.0359, -0.1303, -0.1831, -0.2752, -0.2682,  0.1715]],
       requires_grad=True)

In [67]:
xmlp = w3(xw)
xmlp.shape, xmlp

(torch.Size([2, 8, 6]),
 tensor([[[-0.0262, -0.0254,  0.0048, -0.0418,  0.0527, -0.0047],
          [-0.0300, -0.0201,  0.0293, -0.0070,  0.1138, -0.0239],
          [-0.0341, -0.0190,  0.0265, -0.0143,  0.0857, -0.0190],
          [-0.0352, -0.0257,  0.0256,  0.0045,  0.1131, -0.0271],
          [-0.0311, -0.0212,  0.0260, -0.0095,  0.1065, -0.0212],
          [-0.0294, -0.0158,  0.0367, -0.0059,  0.1255, -0.0255],
          [-0.0286, -0.0162,  0.0355, -0.0102,  0.1225, -0.0247],
          [-0.0223, -0.0173,  0.0261, -0.0270,  0.1056, -0.0139]],
 
         [[-0.0256, -0.0205,  0.0181, -0.0609,  0.0701,  0.0052],
          [-0.0169, -0.0186,  0.0164, -0.0391,  0.0796, -0.0042],
          [-0.0269, -0.0093,  0.0514, -0.0305,  0.1220, -0.0138],
          [-0.0281,  0.0176,  0.0856, -0.0280,  0.1938, -0.0194],
          [-0.0258, -0.0011,  0.0545, -0.0300,  0.1461, -0.0159],
          [-0.0176, -0.0118,  0.0338, -0.0190,  0.1161, -0.0169],
          [-0.0269, -0.0093,  0.0514, -0.0305,  0

#### Residual Connection 2

In [68]:
x = x + xmlp
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.8379,  1.6008,  1.2979,  0.6881,  1.3790,  0.0888],
          [ 0.7301,  1.2716,  1.0440,  0.6493,  1.7416,  0.0681],
          [ 0.7797,  1.4423,  0.9088,  0.3593,  1.5196,  0.3809],
          [ 0.7659,  1.3036,  1.0760,  0.8034,  1.7710,  0.2306],
          [ 0.7167,  1.3198,  1.1199,  0.6698,  1.7747,  0.1409],
          [ 0.7670,  1.2933,  1.0468,  0.6211,  1.8878, -0.0027],
          [ 0.7300,  1.2580,  0.9710,  0.5675,  1.7353, -0.0322],
          [ 0.6147,  1.5504,  1.3627,  0.7808,  1.9489, -0.1331]],
 
         [[ 0.5204,  1.2886,  1.0034,  0.3728,  1.0917, -0.2108],
          [ 0.4392,  1.1189,  1.1023,  0.5891,  1.2815, -0.1638],
          [ 0.4769,  1.1418,  0.6334,  0.0305,  1.2565,  0.0779],
          [ 0.3776,  1.0044,  0.6903,  0.1678,  1.6722, -0.3522],
          [ 0.3976,  1.0244,  0.8207,  0.2692,  1.5373, -0.2200],
          [ 0.5985,  1.1688,  1.1183,  0.5889,  1.6668, -0.2300],
          [ 0.4769,  1.1418,  0.6334,  0.0305,  1

### Final Layer Normalization

In [69]:
weight_f = nn.Parameter(torch.ones(embed_dim))
eps_f = 1e-6

weight_f, eps_f

(Parameter containing:
 tensor([1., 1., 1., 1., 1., 1.], requires_grad=True),
 1e-06)

In [71]:
x_fp32 = x.float()
norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + eps_f)
norm

tensor([[[ 0.7580,  1.4481,  1.1741,  0.6225,  1.2474,  0.0803],
         [ 0.6910,  1.2034,  0.9880,  0.6145,  1.6482,  0.0644],
         [ 0.7735,  1.4307,  0.9015,  0.3564,  1.5074,  0.3779],
         [ 0.6954,  1.1836,  0.9769,  0.7294,  1.6079,  0.2093],
         [ 0.6576,  1.2109,  1.0275,  0.6145,  1.6283,  0.1293],
         [ 0.6950,  1.1720,  0.9487,  0.5628,  1.7107, -0.0024],
         [ 0.7072,  1.2187,  0.9407,  0.5498,  1.6812, -0.0312],
         [ 0.5001,  1.2614,  1.1087,  0.6353,  1.5856, -0.1083]],

        [[ 0.6138,  1.5198,  1.1834,  0.4397,  1.2875, -0.2486],
         [ 0.4976,  1.2674,  1.2487,  0.6673,  1.4516, -0.1855],
         [ 0.6228,  1.4911,  0.8272,  0.0399,  1.6409,  0.1017],
         [ 0.4323,  1.1501,  0.7904,  0.1922,  1.9147, -0.4033],
         [ 0.4661,  1.2010,  0.9622,  0.3156,  1.8023, -0.2579],
         [ 0.5911,  1.1542,  1.1044,  0.5815,  1.6459, -0.2271],
         [ 0.6228,  1.4911,  0.8272,  0.0399,  1.6409,  0.1017],
         [ 0.2837,  1.3

In [73]:
x = (norm * weight_f).type_as(x)
x

tensor([[[ 0.7580,  1.4481,  1.1741,  0.6225,  1.2474,  0.0803],
         [ 0.6910,  1.2034,  0.9880,  0.6145,  1.6482,  0.0644],
         [ 0.7735,  1.4307,  0.9015,  0.3564,  1.5074,  0.3779],
         [ 0.6954,  1.1836,  0.9769,  0.7294,  1.6079,  0.2093],
         [ 0.6576,  1.2109,  1.0275,  0.6145,  1.6283,  0.1293],
         [ 0.6950,  1.1720,  0.9487,  0.5628,  1.7107, -0.0024],
         [ 0.7072,  1.2187,  0.9407,  0.5498,  1.6812, -0.0312],
         [ 0.5001,  1.2614,  1.1087,  0.6353,  1.5856, -0.1083]],

        [[ 0.6138,  1.5198,  1.1834,  0.4397,  1.2875, -0.2486],
         [ 0.4976,  1.2674,  1.2487,  0.6673,  1.4516, -0.1855],
         [ 0.6228,  1.4911,  0.8272,  0.0399,  1.6409,  0.1017],
         [ 0.4323,  1.1501,  0.7904,  0.1922,  1.9147, -0.4033],
         [ 0.4661,  1.2010,  0.9622,  0.3156,  1.8023, -0.2579],
         [ 0.5911,  1.1542,  1.1044,  0.5815,  1.6459, -0.2271],
         [ 0.6228,  1.4911,  0.8272,  0.0399,  1.6409,  0.1017],
         [ 0.2837,  1.3